# LIVE FLIGHT DATA DASHBOARD
## Initial Setup
### Imports

In [57]:
import pandas as pd
import matplotlib.pyplot as plt
from math import radians, cos, sin, sqrt, atan2
from pathlib import Path
from pyopensky.rest import REST

### Connect to api, setup initial Dataframe

In [58]:
api = REST()
flights_df = api.states()
print(flights_df.head())

   icao24 callsign origin_country             last_position  \
0  39de4e  TVF49FP         France 2026-03-15 06:57:02+00:00   
1  4b1815   SWR82P    Switzerland 2026-03-15 06:57:02+00:00   
2  4b1817  SWR2121    Switzerland 2026-03-15 06:57:00+00:00   
3  80162d  AXB1047          India 2026-03-15 06:55:37+00:00   
4  4b1819  SWR542D    Switzerland 2026-03-15 06:57:02+00:00   

                  timestamp  longitude  latitude  altitude onground  \
0 2026-03-15 06:57:02+00:00    -5.7661   40.5505  11269.98    False   
1 2026-03-15 06:57:02+00:00    15.0359   45.5897   11887.2    False   
2 2026-03-15 06:57:02+00:00      8.554   47.4541      <NA>     True   
3 2026-03-15 06:55:37+00:00    77.1016   28.5634      <NA>     True   
4 2026-03-15 06:57:02+00:00     4.5483   48.6512   11582.4    False   

   groundspeed   track  vertical_rate sensors  geoaltitude squawk    spi  \
0       244.64  221.16            0.0    None     11513.82   7655  False   
1       226.78  131.14            0.0    N

### Test api connection, quick view data

In [59]:
print(flights_df.columns)
print(flights_df.info())
print(flights_df.describe())
print(flights_df.shape)

Index(['icao24', 'callsign', 'origin_country', 'last_position', 'timestamp',
       'longitude', 'latitude', 'altitude', 'onground', 'groundspeed', 'track',
       'vertical_rate', 'sensors', 'geoaltitude', 'squawk', 'spi',
       'position_source'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 4660 entries, 0 to 4659
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype             
---  ------           --------------  -----             
 0   icao24           4660 non-null   string[pyarrow]   
 1   callsign         4660 non-null   string[pyarrow]   
 2   origin_country   4660 non-null   string[pyarrow]   
 3   last_position    4619 non-null   datetime64[s, UTC]
 4   timestamp        4660 non-null   datetime64[s, UTC]
 5   longitude        4619 non-null   double[pyarrow]   
 6   latitude         4619 non-null   double[pyarrow]   
 7   altitude         4122 non-null   double[pyarrow]   
 8   onground         4660 non-null   bool[pyarrow]     
 9   gr

### Data exploration
#### Location variables for testing:

In [60]:
GLIDE_LAT, GLIDE_LON = 42.347, -123.438
ATL_LAT, ATL_LON = 33.636667, -84.428056
DUBAI_LAT, DUBAI_LON = 25.252778, 55.364444
TOKYO_LAT, TOKYO_LON = 35.553333, 139.781111
CHI_LAT, CHI_LON = 41.978611, -87.904722

#### Distance calculation:

In [61]:
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    c = 2*atan2(sqrt(a), sqrt(1-a))
    return R * c 

#### Dataframe preparation:

In [62]:
flights_df = api.states()
flights_df = flights_df.dropna(subset=['latitude', 'longitude'])
flights_df['geoaltitude_ft'] = (flights_df['geoaltitude'] * 3.281).round()
flights_df['groundspeed_mph'] = (flights_df['groundspeed'] * 2.237).round()

def get_flights_within_100_miles(api, lat, lon, radius_miles=100):

    flights_df['distance_miles'] = flights_df.apply(
        lambda row: haversine(lat, lon, row['latitude'], row['longitude']), axis=1
    )

    flights_within_100_miles = flights_df[flights_df['distance_miles'] <= radius_miles]
    return flights_within_100_miles[['callsign', 'geoaltitude_ft', 'groundspeed_mph', 'latitude', 'longitude', 'distance_miles']]

print(get_flights_within_100_miles(flights_df, GLIDE_LAT, GLIDE_LON))


     callsign  geoaltitude_ft  groundspeed_mph  latitude  longitude  \
82     ASA706         34052.0            605.0   41.6993  -122.8478   
584   SKW6203         32952.0            385.0   42.6162   -121.898   
1356  SKW5579         34977.0            410.0   41.5287   -122.058   
1621  SKW6396          5000.0            205.0   42.5602  -122.9708   
2351   N1273A            <NA>              8.0   42.3765  -122.8741   

      distance_miles  
82         54.041170  
584        80.645947  
1356       90.703390  
1621       28.004805  
2351       28.861358  


#### Location and distance testing:

In [63]:
atlanta_flights = get_flights_within_100_miles(flights_df, ATL_LAT, ATL_LON)
chicago_flights = get_flights_within_100_miles(flights_df, CHI_LAT, CHI_LON)
dubai_flights = get_flights_within_100_miles(flights_df, DUBAI_LAT, DUBAI_LON)
tokyo_flights = get_flights_within_100_miles(flights_df, TOKYO_LAT, TOKYO_LON)
glide_flights = get_flights_within_100_miles(flights_df, GLIDE_LAT, GLIDE_LON)

print(len(flights_df), "Flights in Air Worldwide")
print(len(glide_flights), "Flights Over Glide")

4622 Flights in Air Worldwide
5 Flights Over Glide


In [64]:
print("Average Altitude", ((flights_df['geoaltitude_ft']).mean()).__round__(2))
print("Average Groundspeed", ((flights_df['groundspeed_mph']).mean()).__round__(2))

Average Altitude 26056.83
Average Groundspeed 389.63
